# Dataset Preparation — Multilingual Voice Language Classifier

Run this notebook in Google Colab after uploading your finalized candidate CSV.

It will:
1. Keep a copy of the uploaded original in Colab's local runtime.
2. validate category names and derive the final language label,
3. audit category counts, blank text, duplicates, and conflicting labels,
4. create group-aware train/validation/test splits,
5. export processed artifacts, reports, and a manifest,
6. optionally push the outputs to this GitHub repository.

**Important:** the repository is public right now. Confirm redistribution rights before uploading any dataset text. Large files require Git LFS or a suitable dataset host. This notebook does not invent missing rows.


## 1. Install dependencies


In [ ]:
!pip -q install pandas numpy scikit-learn


## 2. Upload the final CSV


In [ ]:
from google.colab import files
uploaded = files.upload()
INPUT_FILE = next(iter(uploaded.keys()))
print("Using:", INPUT_FILE)


## 3. Load project config


In [ ]:
import pandas as pd, numpy as np, json, hashlib, re, unicodedata, os, shutil, subprocess, zipfile
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

REPO_URL = "https://github.com/abhinavnautiyalDS/multilingual-voice-language-classifier.git"
OUTPUT = Path("/content/prepared_dataset")
OUTPUT.mkdir(parents=True, exist_ok=True)

CATEGORY_TO_LABEL = {
  "english_only": "en-IN",
  "hindi_only": "hi-IN",
  "hindi_english_mixed_script": "hi-IN",
  "punjabi_only": "pa-IN",
  "punjabi_hindi_mixed_script": "pa-IN",
  "punjabi_english_mixed_script": "pa-IN",
  "punjabi_hindi_english_mixed_script": "pa-IN",
  "bengali_only": "bn-IN",
  "bengali_english_mixed_script": "bn-IN",
  "gujarati_only": "gu-IN",
  "gujarati_english_mixed_script": "gu-IN",
  "kannada_only": "kn-IN",
  "kannada_english_mixed_script": "kn-IN",
  "malayalam_only": "ml-IN",
  "malayalam_english_mixed_script": "ml-IN",
  "marathi_only": "mr-IN",
  "marathi_english_mixed_script": "mr-IN",
  "marathi_hindi_mixed_script": "mr-IN",
  "telugu_only": "te-IN",
  "telugu_english_mixed_script": "te-IN",
  "assamese_only": "as-IN",
  "assamese_english_mixed_script": "as-IN",
  "tamil_only": "ta-IN",
  "tamil_english_mixed_script": "ta-IN",
  "unsupported_language_only": "none",
  "unsupported_language_english_mixed_script": "none"
}

CATEGORY_TARGETS = {
  "english_only": 20000,
  "hindi_only": 20000,
  "hindi_english_mixed_script": 20000,
  "punjabi_only": 20000,
  "punjabi_hindi_mixed_script": 10000,
  "punjabi_english_mixed_script": 5000,
  "punjabi_hindi_english_mixed_script": 5000,
  "bengali_only": 20000,
  "bengali_english_mixed_script": 20000,
  "gujarati_only": 20000,
  "gujarati_english_mixed_script": 20000,
  "kannada_only": 20000,
  "kannada_english_mixed_script": 20000,
  "malayalam_only": 20000,
  "malayalam_english_mixed_script": 20000,
  "marathi_only": 20000,
  "marathi_english_mixed_script": 10000,
  "marathi_hindi_mixed_script": 10000,
  "telugu_only": 20000,
  "telugu_english_mixed_script": 20000,
  "assamese_only": 20000,
  "assamese_english_mixed_script": 20000,
  "tamil_only": 20000,
  "tamil_english_mixed_script": 20000,
  "unsupported_language_only": 10000,
  "unsupported_language_english_mixed_script": 10000
}
print("Configured category target total:", sum(CATEGORY_TARGETS.values()))
print("Earlier plan headline was 500,000; this configuration totals 440,000 after your latest reductions.")


## 4. Preserve original upload and inspect columns


In [ ]:
raw_dir = Path("/content/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
original_copy = raw_dir / Path(INPUT_FILE).name
shutil.copy2(INPUT_FILE, original_copy)

df = pd.read_csv(INPUT_FILE, low_memory=False)
print("Input rows/columns:", df.shape)
print(df.columns.tolist())
display(df.head(3))

TEXT_COL = next((c for c in ["text","sentence","utterance"] if c in df.columns), None)
CATEGORY_COL = next((c for c in ["category","language_category"] if c in df.columns), None)
if TEXT_COL is None:
    raise ValueError("No text column found. Expected text, sentence, or utterance.")
if CATEGORY_COL is None:
    raise ValueError("No category column found. Expected category or language_category.")
print("Text column:", TEXT_COL, "| Category column:", CATEGORY_COL)


## 5. Canonicalize and validate


In [ ]:
if "label" in df.columns:
    df["original_label"] = df["label"].astype("string")

df["text"] = df[TEXT_COL].astype("string").fillna("").str.strip()
df["category"] = (df[CATEGORY_COL].astype("string").fillna("").str.strip().str.lower()
                  .str.replace(r"\s+", "_", regex=True))
blank_mask = df["text"].eq("")
blank_rows = int(blank_mask.sum())
df = df.loc[~blank_mask].copy()

unknown = sorted(set(df["category"]) - set(CATEGORY_TO_LABEL))
if unknown:
    pd.DataFrame({"unknown_category": unknown}).to_csv(OUTPUT / "unknown_categories.csv", index=False)
    raise ValueError(f"Unknown categories: {unknown}. Review unknown_categories.csv and update the map explicitly.")

df["label"] = df["category"].map(CATEGORY_TO_LABEL)
def normalize_text(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(s))).strip().casefold()
df["_normalized_text"] = df["text"].map(normalize_text)
df["_text_sha256"] = df["_normalized_text"].map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())

conflict_hashes = set(df.groupby("_text_sha256")["label"].nunique().loc[lambda x: x > 1].index)
if conflict_hashes:
    df[df["_text_sha256"].isin(conflict_hashes)].to_csv(OUTPUT / "conflicting_duplicate_texts.csv", index=False)
    raise ValueError(f"{len(conflict_hashes)} normalized texts have conflicting labels. Inspect conflicting_duplicate_texts.csv.")

before_dedup = len(df)
df = df.drop_duplicates("_text_sha256", keep="first").copy()
print("Blank rows removed:", blank_rows)
print("Duplicates removed:", before_dedup - len(df))
print("Rows after deduplication:", len(df))


## 6. Category target audit


In [ ]:
actual = df["category"].value_counts().rename_axis("category").reset_index(name="actual_rows")
targets = pd.DataFrame({"category": list(CATEGORY_TARGETS), "target_rows": list(CATEGORY_TARGETS.values())})
audit = targets.merge(actual, on="category", how="left")
audit["actual_rows"] = audit["actual_rows"].fillna(0).astype(int)
audit["difference"] = audit["actual_rows"] - audit["target_rows"]
audit["target_met"] = audit["actual_rows"] == audit["target_rows"]
display(audit)
audit.to_csv(OUTPUT / "category_target_audit.csv", index=False)
print("Actual total after dedup:", f"{len(df):,}")
print("Configured target total:", f"{sum(CATEGORY_TARGETS.values()):,}")
print("Do not silently sample again here: you've finalized counts. Investigate any difference in this audit.")


## 7. Create leakage groups and group-safe splits


In [ ]:
def first_nonempty(row, columns):
    for col in columns:
        if col in row.index and pd.notna(row[col]):
            value = str(row[col]).strip()
            if value and value.lower() not in {"nan","none","null"}:
                return value
    return ""

def make_group(row):
    source = first_nonempty(row, ["source_dataset","source","source_file"]) or "unknown_source"
    explicit = first_nonempty(row, ["source_group_id","group_id"])
    if explicit:
        return f"source::{source}::{explicit}"
    synthetic = str(row.get("is_synthetic","")).strip().lower() in {"true","1","yes"}
    template = first_nonempty(row, ["template_id"])
    generator = first_nonempty(row, ["generator"])
    if synthetic and template:
        return f"template::{source}::{generator}::{template}"
    return f"text::{row['_text_sha256']}"

df["_group_id"] = df.apply(make_group, axis=1)
test_names = {"test","testing","official_test","public_test","test_public"}
if "source_split" in df.columns:
    source_split = df["source_split"].astype("string").fillna("").str.strip().str.lower()
    official_groups = set(df.loc[source_split.isin(test_names), "_group_id"])
else:
    official_groups = set()

official_test = df[df["_group_id"].isin(official_groups)].copy()
pool = df[~df["_group_id"].isin(official_groups)].copy()
if pool["_group_id"].nunique() < 10:
    raise ValueError("Not enough independent groups for the requested group-safe split.")

def take_holdout(frame, n_splits, seed=42):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idx_train, idx_holdout = next(splitter.split(np.zeros((len(frame),1)), frame["label"], frame["_group_id"]))
    return frame.iloc[idx_holdout].copy(), frame.iloc[idx_train].copy()

new_test, remainder = take_holdout(pool, 10)
validation, train = take_holdout(remainder, 9)
test = pd.concat([official_test, new_test], axis=0)

groups = {k:set(v["_group_id"]) for k,v in {"train":train,"validation":validation,"test":test}.items()}
assert groups["train"].isdisjoint(groups["validation"])
assert groups["train"].isdisjoint(groups["test"])
assert groups["validation"].isdisjoint(groups["test"])

print({k: len(v) for k,v in {"train":train,"validation":validation,"test":test}.items()})
print("Group leakage check passed.")


## 8. Save cleaned data, splits, reports and manifest


In [ ]:
# Cleaned/deduplicated final candidate pool (before split)
cleaned = df.drop(columns=["_normalized_text","_group_id"], errors="ignore").copy()
cleaned.to_csv(OUTPUT / "final_cleaned_dataset.csv.gz", index=False, compression="gzip")

for split_name, frame in {"train":train,"validation":validation,"test":test}.items():
    frame.drop(columns=["_normalized_text","_group_id"], errors="ignore").to_csv(
        OUTPUT / f"{split_name}.csv.gz", index=False, compression="gzip"
    )

label_counts = []
for split_name, frame in {"train":train,"validation":validation,"test":test}.items():
    for label, count in frame["label"].value_counts().items():
        label_counts.append({"split":split_name,"label":label,"rows":int(count)})
pd.DataFrame(label_counts).to_csv(OUTPUT / "label_counts_by_split.csv", index=False)

manifest = {
    "input_filename": Path(INPUT_FILE).name,
    "input_rows": int(pd.read_csv(INPUT_FILE, usecols=[TEXT_COL], low_memory=False).shape[0]),
    "blank_rows_removed": blank_rows,
    "duplicates_removed": int(before_dedup - len(df)),
    "rows_after_dedup": int(len(df)),
    "split_rows": {"train":int(len(train)),"validation":int(len(validation)),"test":int(len(test))},
    "category_targets": CATEGORY_TARGETS,
    "configured_target_total": int(sum(CATEGORY_TARGETS.values())),
    "previous_plan_headline_total": 500000,
    "label_map": CATEGORY_TO_LABEL,
    "seed": 42,
    "group_leakage_check": "passed",
    "note": "Review source licenses before redistributing dataset text."
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Artifacts:")
for p in sorted(OUTPUT.iterdir()):
    print(p.name, f"{p.stat().st_size/1024/1024:.2f} MiB")


## 9. Download prepared artifacts


In [ ]:
zip_path = Path("/content/prepared_dataset.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT.iterdir():
        if p.is_file():
            zf.write(p, arcname=f"data/processed/{p.name}")
files.download(str(zip_path))


## 10. Optional: push artifacts to GitHub using a Colab Secret


In [ ]:
# Push the original upload to data/raw, cleaned/deduplicated pool to data/interim,
# and train/validation/test plus audit metadata to data/processed.
# These files are uploaded only after the license confirmation above.
raw_dest = repo_dir / "data" / "raw"
interim_dest = repo_dir / "data" / "interim"
processed_dest = repo_dir / "data" / "processed"
for folder in [raw_dest, interim_dest, processed_dest]:
    folder.mkdir(parents=True, exist_ok=True)

shutil.copy2(original_copy, raw_dest / Path(INPUT_FILE).name)
for p in OUTPUT.iterdir():
    if not p.is_file():
        continue
    if p.name == "final_cleaned_dataset.csv.gz":
        shutil.copy2(p, interim_dest / p.name)
    else:
        shutil.copy2(p, processed_dest / p.name)

# Track CSVs with Git LFS; install Git LFS in this Colab runtime first.
subprocess.run(["git","-C",str(repo_dir),"lfs","track","data/raw/*","data/interim/*.csv.gz","data/processed/*.csv.gz"], check=True)
subprocess.run(["git","-C",str(repo_dir),"add",".gitattributes","data/raw","data/interim","data/processed"], check=True)
subprocess.run(["git","-C",str(repo_dir),"config","user.name","Colab Dataset Prep"], check=True)
subprocess.run(["git","-C",str(repo_dir),"config","user.email","colab-dataset-prep@users.noreply.github.com"], check=True)
subprocess.run(["git","-C",str(repo_dir),"commit","-m","Add raw, cleaned, and prepared dataset artifacts"], check=True)
subprocess.run(["git","-C",str(repo_dir),"push","origin","HEAD"], check=True, env=env)
print("Raw, interim, and processed dataset artifacts pushed to:", REPO_URL)
